# 🛩️ Пример работы с моделью Ultrastick-25e

Этот notebook демонстрирует использование линейной продольной модели самолета Ultrastick-25e в среде TensorAeroSpace.

## 📋 Что мы будем делать:
1. Импортируем необходимые библиотеки
2. Настроим временные параметры и опорный сигнал
3. Создадим и инициализируем среду
4. Выполним один шаг симуляции
5. Проанализируем результаты

## 📚 Импорт библиотек

Загружаем все необходимые модули для работы с моделью самолета:

In [ ]:
# Основные библиотеки
import gymnasium as gym
import numpy as np

# Импорт TensorAeroSpace для регистрации окружений
import tensoraerospace

# Модули TensorAeroSpace
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

## ⚙️ Настройка параметров симуляции

Определяем временные параметры и создаем опорный сигнал для управления углом тангажа:

In [ ]:
# Параметры дискретизации
dt = 0.01  # Шаг дискретизации (секунды)

# Генерация временного периода
tp = generate_time_period(tn=20, dt=dt)  # 20 секунд симуляции
tps = convert_tp_to_sec_tp(tp, dt=dt)    # Преобразование в секунды
number_time_steps = len(tp)             # Общее количество временных шагов

# Создание ступенчатого опорного сигнала для угла тангажа (theta)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10, output_rad=True), 
    [1, -1]
)

print(f"📊 Параметры симуляции:")
print(f"   • Время симуляции: {tp[-1]:.1f} сек")
print(f"   • Шаг дискретизации: {dt} сек")
print(f"   • Количество шагов: {number_time_steps}")
print(f"   • Форма опорного сигнала: {reference_signals.shape}")

## 🚀 Создание и инициализация среды

Создаем среду Ultrastick-25e с заданными параметрами и выполняем сброс:

In [ ]:
# Создание среды Ultrastick-25e
env = gym.make(
    "LinearLongitudinalUltrastick-v0",
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0], [0]],  # [u, w, q, theta, h]
    reference_signal=reference_signals,
    tracking_states=["theta"]
)

# Инициализация среды
state, info = env.reset()

print(f"✅ Среда успешно создана и инициализирована!")
print(f"📈 Начальное состояние: {state.flatten()}")
print(f"🎯 Форма пространства состояний: {env.observation_space.shape}")
print(f"🎮 Форма пространства действий: {env.action_space.shape}")
print(f"📊 Отслеживаемые состояния: {env.unwrapped.tracking_states}")
print(f"📋 Пространство состояний: {env.unwrapped.state_space}")
print(f"📤 Пространство выходов: {env.unwrapped.output_space}")

## 🎮 Выполнение шага симуляции

Применяем управляющее воздействие и наблюдаем реакцию системы:

In [ ]:
# Применение управляющего воздействия
# Действие должно быть одномерным массивом
control_input = np.array([1.0], dtype=np.float32)  # Форма (1,) для среды

# Выполнение одного шага симуляции
state, reward, terminated, truncated, info = env.step(control_input)

print(f"🎯 Управляющее воздействие: {control_input[0]:.2f} градусов")
print(f"📊 Новое состояние: {state.flatten()}")
theta_idx = env.unwrapped.state_space.index("theta")
q_idx = env.unwrapped.state_space.index("q")
print(f"   • Угол тангажа (theta): {np.rad2deg(state[theta_idx, 0]):.4f} град")
print(f"   • Угловая скорость (q): {np.rad2deg(state[q_idx, 0]):.4f} град/с")
# Обработка награды (может быть массивом или скаляром)
reward_value = reward[0] if isinstance(reward, np.ndarray) else reward
print(f"🏆 Награда: {reward_value:.6f}")
print(f"🔚 Завершено: {terminated}")
print(f"⏰ Прервано: {truncated}")